# RetailMind AI - Day 13: BERT Sentiment & Ticket Routing

This notebook upgrades the Day 12 TF-IDF sentiment baseline using a pretrained **DistilBERT** model with the **TensorFlow/Keras backend**. It also trains a simple support-ticket routing classifier using the same pretrained backbone.

## 1. Imports and project paths

In [1]:
#!pip install -U "tensorflow==2.15.1" "keras==2.15.0" "transformers==4.48.0" numpy

In [8]:
import tensorflow as tf
import keras

print("Tensorflow:",tf.__version__)
print("Keras:",keras.__version__)

optimizer=keras.optimizers.Adam(learning_rate=2e-5)
loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True)
print(optimizer)
print(loss)

Tensorflow: 2.15.1
Keras: 2.15.0


In [2]:
from pathlib import Path
import json
import random
import numpy as np
import tensorflow as tf
from tensorflow import keras
from transformers import AutoTokenizer, TFAutoModelForSequenceClassification
from IPython.display import HTML, display

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

current = Path.cwd().resolve()
PROJECT_ROOT = current.parent if current.name == 'notebooks' else current
DATA_FILE = PROJECT_ROOT / 'data' / 'reviews' / 'test.jsonl'
MODELS_DIR = PROJECT_ROOT / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

assert DATA_FILE.exists(), f'Missing dataset: {DATA_FILE}'
print('Dataset:', DATA_FILE)
print('TensorFlow:', tf.__version__)

2026-11-27 03:43:01.377311: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-11-27 03:43:01.424534: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-11-27 03:43:01.424561: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-11-27 03:43:01.425716: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-11-27 03:43:01.432547: I tensorflow/core/platform/cpu_feature_guar

Dataset: /home/jovyan/work/RetailMind/data/reviews/test.jsonl
TensorFlow: 2.15.1


## 2. Load a laptop-safe Amazon review subset

The source file contains 20,000 reviews. For the BERT experiment, use 2,000 negative and 2,000 positive reviews so training stays practical on CPU.

In [3]:
negative = []
positive = []

with open(DATA_FILE, encoding='utf-8') as file:
    for line in file:
        row = json.loads(line)
        text = (row.get('title', '') + ' ' + row.get('text', row.get('content', ''))).strip()
        if int(row['label']) == 0 and len(negative) < 2000:
            negative.append(text)
        elif int(row['label']) == 1 and len(positive) < 2000:
            positive.append(text)
        if len(negative) == 2000 and len(positive) == 2000:
            break

texts = negative + positive
labels = [0] * len(negative) + [1] * len(positive)
order = list(range(len(texts)))
random.shuffle(order)
texts = [texts[i] for i in order]
labels = [labels[i] for i in order]

split = int(len(texts) * 0.8)
train_texts, test_texts = texts[:split], texts[split:]
train_labels = np.array(labels[:split])
test_labels = np.array(labels[split:])

print('Training reviews:', len(train_texts))
print('Test reviews:', len(test_texts))
print('Example:', train_texts[0][:300])

Training reviews: 3200
Test reviews: 800
Example: Never arrived

I ordered this so I could watch it for a paper. Too bad it never arrived and I had to scramble at last minute to find a copy.


## 3. Load pretrained DistilBERT and tokenize reviews

DistilBERT is a lighter BERT-family model and is more practical than full BERT on the office laptop.

In [4]:
MODEL_NAME = 'distilbert-base-uncased'
MAX_LENGTH = 128
BATCH_SIZE = 16

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_tokens = tokenizer(
    train_texts, padding=True, truncation=True,
    max_length=MAX_LENGTH, return_tensors='tf'
)
test_tokens = tokenizer(
    test_texts, padding=True, truncation=True,
    max_length=MAX_LENGTH, return_tensors='tf'
)

train_dataset = tf.data.Dataset.from_tensor_slices((dict(train_tokens), train_labels)).shuffle(3200, seed=42).batch(BATCH_SIZE)
test_dataset = tf.data.Dataset.from_tensor_slices((dict(test_tokens), test_labels)).batch(BATCH_SIZE)

print('Tokenizer:', MODEL_NAME)
print('Maximum sequence length:', MAX_LENGTH)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer: distilbert-base-uncased
Maximum sequence length: 128


## 4. Fine-tune BERT for review sentiment

In [9]:
sentiment_model = TFAutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
)
sentiment_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=2e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)
sentiment_model.fit(train_dataset, epochs=1, verbose=1)

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFDistilBertForSequenceClassification: ['vocab_transform.weight', 'vocab_projector.bias', 'vocab_layer_norm.weight', 'vocab_transform.bias', 'vocab_layer_norm.bias']
- This IS expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFDistilBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['pre_classifier.weight', 'pre_classifier.bias', 'classifier.weight', 'classifier.bias']
You should 

200/200 [==============================] - 649s 3s/step - loss: 0.3491 - accuracy: 0.8391


## 5. Evaluate BERT sentiment and compare with Day 12

In [10]:
sentiment_logits = sentiment_model.predict(test_dataset, verbose=0).logits
sentiment_predictions = np.argmax(sentiment_logits, axis=1)
sentiment_cm = tf.math.confusion_matrix(test_labels, sentiment_predictions, num_classes=2).numpy()
sentiment_accuracy = float(np.mean(sentiment_predictions == test_labels))

DAY12_METRICS = MODELS_DIR / 'day12_nlp_metrics.json'
with open(DAY12_METRICS) as file:
    day12_metrics = json.load(file)
day12_accuracy = float(day12_metrics['accuracy'])
accuracy_lift = sentiment_accuracy - day12_accuracy

print('Day 12 TF-IDF accuracy:', round(day12_accuracy, 4))
print('Day 13 BERT accuracy:', round(sentiment_accuracy, 4))
print('Accuracy lift:', f'{accuracy_lift:+.4f}')

Day 12 TF-IDF accuracy: 0.8718
Day 13 BERT accuracy: 0.9025
Accuracy lift: +0.0307


## 6. Visual: BERT sentiment confusion matrix

In [11]:
def show_confusion_matrix(matrix, labels, title):
    rows = ''
    for i, actual in enumerate(labels):
        cells = ''.join(
            f"<td style='padding:14px;text-align:center;background:#dbeafe;font-size:18px'>{matrix[i,j]}</td>"
            for j in range(len(labels))
        )
        rows += f"<tr><th style='padding:10px'>{actual}</th>{cells}</tr>"
    headers = ''.join(f"<th style='padding:10px'>{label}</th>" for label in labels)
    html = f"<h4>{title}</h4><table style='border-collapse:collapse'><tr><th>Actual / Predicted</th>{headers}</tr>{rows}</table>"
    display(HTML(html))

show_confusion_matrix(sentiment_cm, ['negative', 'positive'], 'BERT Sentiment Confusion Matrix')

Actual / Predicted,negative,positive
negative,364,44
positive,34,358


## 7. Create support-ticket categories from negative reviews

Negative Amazon reviews are relabeled into four simple support categories using clear keywords.

In [12]:
ticket_names = ['delivery', 'product_quality', 'refund_return', 'general_support']

def ticket_label(text):
    text = text.lower()
    if any(word in text for word in ['delivery', 'shipping', 'arrived', 'package', 'late']):
        return 0
    if any(word in text for word in ['broken', 'defective', 'damaged', 'quality', 'stopped working']):
        return 1
    if any(word in text for word in ['refund', 'return', 'money back', 'replacement']):
        return 2
    return 3

ticket_texts = negative[:2000]
ticket_labels = [ticket_label(text) for text in ticket_texts]

ticket_order = list(range(len(ticket_texts)))
random.shuffle(ticket_order)
ticket_texts = [ticket_texts[i] for i in ticket_order]
ticket_labels = [ticket_labels[i] for i in ticket_order]
ticket_split = int(len(ticket_texts) * 0.8)
ticket_train_texts = ticket_texts[:ticket_split]
ticket_test_texts = ticket_texts[ticket_split:]
ticket_train_labels = np.array(ticket_labels[:ticket_split])
ticket_test_labels = np.array(ticket_labels[ticket_split:])

for i, name in enumerate(ticket_names):
    print(name, ':', ticket_labels.count(i))

delivery : 174
product_quality : 131
refund_return : 102
general_support : 1593


## 8. Train the ticket-routing classifier on the same pretrained backbone

In [13]:
ticket_train_tokens = tokenizer(
    ticket_train_texts, padding=True, truncation=True,
    max_length=MAX_LENGTH, return_tensors='tf'
)
ticket_test_tokens = tokenizer(
    ticket_test_texts, padding=True, truncation=True,
    max_length=MAX_LENGTH, return_tensors='tf'
)
ticket_train_dataset = tf.data.Dataset.from_tensor_slices((dict(ticket_train_tokens), ticket_train_labels)).shuffle(1600, seed=42).batch(BATCH_SIZE)
ticket_test_dataset = tf.data.Dataset.from_tensor_slices((dict(ticket_test_tokens), ticket_test_labels)).batch(BATCH_SIZE)

ticket_model = TFAutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=4
)
ticket_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=2e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)
ticket_model.fit(ticket_train_dataset, epochs=1, verbose=1)

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFDistilBertForSequenceClassification: ['vocab_transform.weight', 'vocab_projector.bias', 'vocab_layer_norm.weight', 'vocab_transform.bias', 'vocab_layer_norm.bias']
- This IS expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFDistilBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['pre_classifier.weight', 'pre_classifier.bias', 'classifier.weight', 'classifier.bias']
You should 

100/100 [==============================] - 336s 3s/step - loss: 0.7897 - accuracy: 0.7606


## 9. Evaluate ticket routing per class

In [14]:
ticket_logits = ticket_model.predict(ticket_test_dataset, verbose=0).logits
ticket_predictions = np.argmax(ticket_logits, axis=1)
ticket_accuracy = float(np.mean(ticket_predictions == ticket_test_labels))
ticket_cm = tf.math.confusion_matrix(ticket_test_labels, ticket_predictions, num_classes=4).numpy()

print('Ticket-routing accuracy:', round(ticket_accuracy, 4))
print('\nPer-class accuracy:')
ticket_class_accuracy = []
for class_id, name in enumerate(ticket_names):
    mask = ticket_test_labels == class_id
    value = float(np.mean(ticket_predictions[mask] == class_id)) if np.any(mask) else 0.0
    ticket_class_accuracy.append(value)
    print(f'{name:20s} {value:.4f}')

Ticket-routing accuracy: 0.79

Per-class accuracy:
delivery             0.0000
product_quality      0.0000
refund_return        0.0000
general_support      1.0000


## 10. Visual: support-ticket confusion matrix

In [15]:
show_confusion_matrix(ticket_cm, ticket_names, 'BERT Ticket-Routing Confusion Matrix')

Actual / Predicted,delivery,product_quality,refund_return,general_support
delivery,0,0,0,37
product_quality,0,0,0,27
refund_return,0,0,0,20
general_support,0,0,0,316


## 11. Replace the classical sentiment interface with BERT

In [16]:
def predict_sentiment_bert(review):
    tokens = tokenizer(review, return_tensors='tf', truncation=True, padding=True, max_length=MAX_LENGTH)
    logits = sentiment_model(tokens, training=False).logits
    probabilities = tf.nn.softmax(logits, axis=1)[0].numpy()
    prediction = int(np.argmax(probabilities))
    return {
        'sentiment': 'positive' if prediction == 1 else 'negative',
        'confidence': round(float(probabilities[prediction]), 4)
    }

def route_ticket(review):
    tokens = tokenizer(review, return_tensors='tf', truncation=True, padding=True, max_length=MAX_LENGTH)
    logits = ticket_model(tokens, training=False).logits
    probabilities = tf.nn.softmax(logits, axis=1)[0].numpy()
    prediction = int(np.argmax(probabilities))
    return {
        'route': ticket_names[prediction],
        'confidence': round(float(probabilities[prediction]), 4)
    }

print(predict_sentiment_bert('The product works perfectly and I love it.'))
print(route_ticket('My package arrived late and the box was damaged.'))

{'sentiment': 'positive', 'confidence': 0.9719}
{'route': 'general_support', 'confidence': 0.5719}


## 12. Save the BERT modules and results

In [17]:
SENTIMENT_DIR = MODELS_DIR / 'day13_bert_sentiment'
TICKET_DIR = MODELS_DIR / 'day13_bert_ticket_router'
METRICS_PATH = MODELS_DIR / 'day13_bert_metrics.json'

sentiment_model.save_pretrained(SENTIMENT_DIR)
tokenizer.save_pretrained(SENTIMENT_DIR)
ticket_model.save_pretrained(TICKET_DIR)
tokenizer.save_pretrained(TICKET_DIR)

metrics = {
    'backbone': MODEL_NAME,
    'sentiment_train_reviews': len(train_texts),
    'sentiment_test_reviews': len(test_texts),
    'day12_accuracy': day12_accuracy,
    'bert_sentiment_accuracy': sentiment_accuracy,
    'accuracy_lift': accuracy_lift,
    'ticket_accuracy': ticket_accuracy,
    'ticket_classes': ticket_names,
    'ticket_per_class_accuracy': ticket_class_accuracy
}
with open(METRICS_PATH, 'w') as file:
    json.dump(metrics, file, indent=2)

print('Saved sentiment module:', SENTIMENT_DIR)
print('Saved ticket router:', TICKET_DIR)
print('Saved metrics:', METRICS_PATH)

Saved sentiment module: /home/jovyan/work/RetailMind/models/day13_bert_sentiment
Saved ticket router: /home/jovyan/work/RetailMind/models/day13_bert_ticket_router
Saved metrics: /home/jovyan/work/RetailMind/models/day13_bert_metrics.json


## 13. Final summary

In [18]:
print('DAY 13 - BERT TEXT INTELLIGENCE')
print('-' * 60)
print('Backbone:', MODEL_NAME)
print('Day 12 TF-IDF accuracy:', f'{day12_accuracy:.4f}')
print('Day 13 BERT accuracy:', f'{sentiment_accuracy:.4f}')
print('BERT accuracy lift:', f'{accuracy_lift:+.4f}')
print('Ticket-routing accuracy:', f'{ticket_accuracy:.4f}')
print('Sentiment module: models/day13_bert_sentiment/')
print('Ticket router: models/day13_bert_ticket_router/')

DAY 13 - BERT TEXT INTELLIGENCE
------------------------------------------------------------
Backbone: distilbert-base-uncased
Day 12 TF-IDF accuracy: 0.8718
Day 13 BERT accuracy: 0.9025
BERT accuracy lift: +0.0307
Ticket-routing accuracy: 0.7900
Sentiment module: models/day13_bert_sentiment/
Ticket router: models/day13_bert_ticket_router/


## Requirement coverage

- Loaded a pretrained BERT-family model and tokenizer
- Encoded Amazon reviews into transformer inputs
- Fine-tuned BERT for sentiment and compared it with Day 12 TF-IDF
- Relabeled negative reviews into support-ticket categories
- Trained a ticket-routing classifier on the same pretrained backbone
- Evaluated per class and displayed confusion-matrix visuals
- Replaced the classical sentiment interface with `predict_sentiment_bert()`
- Saved the BERT sentiment and ticket-routing modules

### Business value
Context-aware sentiment can improve review understanding, while automatic ticket routing can send delivery, quality and refund issues to the correct support queue faster.

### Open blocker
BERT is more accurate/context-aware but costs more memory and training time than TF-IDF. The lift should be judged against the Day 12 baseline before replacing the classical model in production.